# 10 — Policy Optimization Mechanics

## Objective
Implement a compact Proximal Policy Optimization (PPO)-style clipped policy-gradient update and distinguish the demonstration from full Large Language Model (LLM) Reinforcement Learning from Human Feedback (RLHF).

## Background
RLHF (Reinforcement Learning from Human Feedback) typically includes preference collection, reward modeling, and policy optimization. PPO (Proximal Policy Optimization) limits how rapidly a policy changes relative to an earlier policy. This CPU example uses a small discrete bandit, not a language model.

### Proximal Policy Optimization (PPO) clipped surrogate

$$L^{\mathrm{CLIP}}(\theta)=\mathbb E_t\left[\min\left(\rho_t(\theta)A_t,\operatorname{clip}(\rho_t(\theta),1-\epsilon,1+\epsilon)A_t\right)\right]$$

The importance ratio is $\rho_t=\pi_\theta(a_t\mid s_t)/\pi_{\mathrm{old}}(a_t\mid s_t)$, $A_t$ is an advantage estimate, and $\epsilon$ defines the surrogate's clipping interval. Clipping removes the objective's incentive to increase a favorable ratio beyond that interval; it does **not** impose a hard bound on every policy change. This surrogate is only one component of a full PPO system.

### Explore a clipped policy update

**Procedure.** Optimize a synthetic three-action bandit with a PPO-style clipped objective and a fixed reward vector. Check the boundary gradients separately, because clipping can be rarely activated in an otherwise successful run.

**Reading the result.** Compare exact expected reward under the starting and final action distributions. The bandit rewards are known by construction and do not come from notebook 09's reward model. This is an algorithm demonstration, not full Reinforcement Learning from Human Feedback (RLHF) for language models.

In [1]:
# A small three-action synthetic policy illustrates the clipped PPO objective.
# It does not train a language model or learn a biomedical reward function.
import torch

torch.manual_seed(42)
logits = torch.nn.Parameter(torch.zeros(3))
optimizer = torch.optim.Adam([logits], lr=0.08)
action_reward = torch.tensor([0.0, 1.0, 0.2])
clip_epsilon = 0.2
initial_probability = torch.softmax(logits.detach(), dim=0)
initial_expected_reward = float((initial_probability * action_reward).sum().item())

# Verify the clipped surrogate's zero gradient in the two outward directions.
# The third ratio remains inside the clipping interval and retains a gradient.
test_ratio = torch.tensor([1.5, 0.5, 1.0], requires_grad=True)
test_advantage = torch.tensor([1.0, -1.0, 1.0])
test_surrogate = torch.minimum(
    test_ratio * test_advantage,
    test_ratio.clamp(1 - clip_epsilon, 1 + clip_epsilon) * test_advantage,
)
test_surrogate.sum().backward()
assert torch.allclose(test_ratio.grad, torch.tensor([0.0, 0.0, 1.0]))
print("Boundary-check ratio gradients:", test_ratio.grad.tolist())

policy_history = []
clip_fractions = []
rollout_kl = []
for rollout in range(50):
    # Sample actions from a frozen old policy. Reuse that rollout for three
    # optimization epochs so probability ratios can move away from one.
    old_probability = torch.softmax(logits.detach(), dim=0)
    actions = torch.multinomial(old_probability, num_samples=32, replacement=True)
    advantages = action_reward[actions] - (old_probability * action_reward).sum()
    advantages = advantages.detach()

    for epoch in range(3):
        new_probability = torch.softmax(logits, dim=0)
        probability_ratio = new_probability[actions] / old_probability[actions]
        unclipped = probability_ratio * advantages
        clipped = probability_ratio.clamp(1 - clip_epsilon, 1 + clip_epsilon) * advantages
        policy_loss = -torch.minimum(unclipped, clipped).mean()
        optimizer.zero_grad()
        policy_loss.backward()
        optimizer.step()

        # These diagnostics refer to the policy before this optimizer step.
        outside_interval = ((probability_ratio < 1 - clip_epsilon) |
                            (probability_ratio > 1 + clip_epsilon))
        clip_fractions.append(float(outside_interval.float().mean().item()))
        old_to_new_kl = (old_probability * (old_probability.log() - new_probability.detach().log())).sum()
        rollout_kl.append(float(old_to_new_kl.item()))

    policy_history.append(torch.softmax(logits.detach(), dim=0).tolist())

final_probability = torch.tensor(policy_history[-1])
final_expected_reward = float((final_probability * action_reward).sum().item())
print("Final action probabilities:", [round(float(p), 3) for p in final_probability])
print("Exact expected reward, initial -> final:", round(initial_expected_reward, 4), "->", round(final_expected_reward, 4))
print("Observed fraction of ratios outside clip interval (mean):", round(sum(clip_fractions) / len(clip_fractions), 4))
print("Maximum observed old-to-current KL divergence before an update:", round(max(rollout_kl), 6))
assert float(final_probability[1]) > float(final_probability[0])
assert float(final_probability[1]) > float(final_probability[2])
assert final_expected_reward > initial_expected_reward
assert all(torch.isfinite(torch.tensor(policy_history).flatten()))


Boundary-check ratio gradients: [0.0, 0.0, 1.0]
Final action probabilities: [0.002, 0.996, 0.002]
Exact expected reward, initial -> final: 0.4 -> 0.9963
Observed fraction of ratios outside clip interval (mean): 0.0025
Maximum observed old-to-current KL divergence before an update: 0.012031


### Results

The three-action synthetic policy's exact expected reward rose from 0.4000 to 0.9963 under fixed rewards `[0, 1, 0.2]`; its best-action probability finished near 0.996. The boundary gradient check returned `[0, 0, 1]`. Only 0.25% of sampled ratios were outside the clip interval on average.

### Interpretation

The toy update demonstrates the clipped surrogate's direction and boundary behavior, with clipping exercised rarely in this trajectory.

### Limitations

This CPU bandit is neither biomedical language-model RLHF nor evidence that notebook 09's near-chance reward model is useful.

### Key Observations

Improving known toy reward does not establish improved answer accuracy or alignment.


### Limits of the demonstration

The reward vector is fixed and synthetic. The policy has three logits, no language-model tokens, and no learned biomedical reward. A language-model PPO run also needs text generation, a reference policy, reward-model inference, usually a value model, token-level masking, and rollout management. Notebook 09's reward model did not pass held-out preference discrimination, so it is not used here. The clipped surrogate by itself does not guarantee a strict divergence limit.

## Key observations and technical discussion

The policy's **advantage** is each sampled action's reward minus the old policy's exact expected reward. Positive advantage makes an action more attractive relative to that baseline; reward itself is the fixed bandit value. The importance ratio compares the current policy's probability of a sampled action with the frozen rollout policy. The boundary check demonstrates where the clipped surrogate has no outward gradient, while the observed clip fraction shows how often sampled ratios actually crossed the interval in this run. A full PPO implementation may monitor or penalize Kullback–Leibler (KL) divergence as another safeguard.

Direct Preference Optimization (DPO) uses preferred/rejected response log-probabilities relative to a reference, without this policy-gradient rollout. Group Relative Policy Optimization (GRPO) estimates relative advantages from a group of sampled completions. Both are separate from this bandit demonstration. A weak learned reward can be exploited by a policy that improves the reward score without improving biomedical correctness; notebook 09 found no reliable held-out reward signal.

## Q&A

**Question:** What is clipped by PPO and why?

**Answer:** The importance ratio is clipped inside the surrogate objective to remove incentive for further movement in the favorable direction after a ratio crosses the interval. It is not a hard constraint on every policy update.

### Reproducibility note
Each printed example belongs to one of three categories: an illustrative unit check, a source-backed inspection, or an opt-in model experiment. Only real executed experiments with source/model identifiers and saved outputs can support research findings.